# KAKEN 大型科研費までの道のり — Colab版

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rma-lab/kaken-history/blob/main/kaken-history.ipynb)

所属機関の研究者が**最初の科研費から大型科研費（基盤研究(B)以上）に到達するまで何年かかったか**を、
科研費データベース（KAKEN）の公開データから集計し、4つのPDFを作ります。

- A4縦1枚の**レポート**と所要年数の**ヒストグラム**（どちらも氏名なし）
- 研究者ごとの採択履歴を並べた**ガントチャート**2種（大型あり群／なし群、氏名あり）

## 使い方は3ステップ

1. **KAKENからデータをダウンロード**（appid等は不要）:
   [KAKEN 研究者をさがす](https://nrid.nii.ac.jp/ja/) の詳細検索を開き、研究者情報の **所属機関** に自分の機関名を入れて検索 →
   検索結果画面で **すべて選択 → JSONで出力 → 実行** でJSONを保存（1回1万件まで）
2. **このノートブックのセルを上から順に実行**（途中で 1. のJSONをアップロードします。（任意）のセルは飛ばしてOK）
3. **できあがったPDFをダウンロード**

> **注意**: 検索は研究者情報の「所属機関」欄で行ってください。研究課題情報の「研究機関」欄で
> 検索すると、その機関に所属したことのない他機関の研究分担者まで大量に含まれ、集計が歪みます。

> アップロードしたJSONは、このColabセッション（Googleのクラウド上の一時環境）で処理されます。
> KAKENの公開データですが氏名・研究者番号を含みます。手元だけで処理したい場合は
> [リポジトリ](https://github.com/rma-lab/kaken-history)をcloneしてローカルで実行できます（README参照）。

## 1. 環境の準備

ライブラリと日本語フォント（IPAゴシック）を入れ、コードを取得します。

In [ ]:
# 日本語フォント（IPAゴシック）を導入。apt-get update を先に（古い索引だと404になる）。
# PDF生成はmatplotlibのみでpoppler不要。プレビューはpipだけで入るPyMuPDFを使う。
!apt-get -qq update > /dev/null
!apt-get -qq -y --fix-missing install fonts-ipafont-gothic > /dev/null
!pip -q install matplotlib pymupdf > /dev/null

# 入れたIPAフォントを matplotlib に登録（中国語字形になるNoto CJKは使わない）
import glob, matplotlib.font_manager as fm, matplotlib.pyplot as plt
ipa = glob.glob('/usr/share/fonts/**/ipag*.ttf', recursive=True)
for path in ipa:
    fm.fontManager.addfont(path)
if ipa:
    plt.rcParams['font.family'] = 'IPAGothic'
print('日本語フォント:', 'OK' if ipa else '見つかりません（ランタイムを再起動して再実行）')

# コード一式を取得（再実行しても入れ子にならないよう絶対パスで。既存なら最新に更新）
import os
os.chdir('/content')
if os.path.exists('/content/kaken-history'):
    !cd /content/kaken-history && git pull -q
else:
    !git clone -q https://github.com/rma-lab/kaken-history.git
os.chdir('/content/kaken-history')
print('準備完了:', os.getcwd())

## 2.（任意）Googleドライブをマウント — 大きいJSONの再アップロード回避

**飛ばしてOKです**（その場合、セッションが破棄されるたびにJSONを再アップロードします）。

実行しておくと、初回にアップロードしたJSONがあなたのGoogleドライブに保存され、
次回以降は自動でそこから読み込まれます（170MB級のアップロードが実質1回で済みます）。
実行するとGoogleの**アクセス許可の画面**が出るので、自分のアカウントで許可してください
（保存先はあなた自身のドライブです）。

> 補足: 「ランタイムを**再起動**」ではアップロード済みファイルは消えません。消えるのは
> 「接続解除して**削除**」やアイドル切断でVMごと破棄されたときだけです。

In [ ]:
# 実行するとドライブへのアクセス許可を求められます（あなた自身のドライブに保存します）
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('ドライブをマウントしました')
except Exception as e:
    print('マウントをスキップ:', e)

## 3.（任意）タイトル用ラベル

レポートのタイトルに入れる文字列です（例: `福島大学`）。**空欄のままでもOK**
（機関名なしの汎用タイトルになります）。

分析には影響しません（集計対象はアップロードしたJSONの中身で決まります）。

In [ ]:
LABEL = ""  #@param {type:"string"}
KEY = 'target'

import os, kaken_inst
kaken_inst.INSTITUTIONS[KEY] = LABEL           # タイトル用ラベルを実行時に登録
os.makedirs(f'data/{KEY}', exist_ok=True)
print('タイトルラベル:', LABEL or '（なし＝汎用レポート）')

## 4. データを用意（初回のみアップロード）

次の優先順位でJSONを用意します：
1. 既にこのセッションにある → そのまま使う（再起動後もここに来ることが多い）
2. Googleドライブに保存済み → そこからコピー（**アップロード不要**）
3. どちらも無ければ → ブラウザからアップロード（＋ドライブがあれば次回用に保存）

In [ ]:
import os, shutil
DEST = f'data/{KEY}/researchers.json'
DRIVE_DIR = '/content/drive/MyDrive/kaken'
DRIVE = f'{DRIVE_DIR}/researchers.json'

def size_mb(p): return f'{os.path.getsize(p)/1e6:.0f}MB'

if os.path.exists(DEST):
    print('既にデータあり（読み込み不要）:', DEST, size_mb(DEST))
elif os.path.exists(DRIVE):
    shutil.copy(DRIVE, DEST)
    print('ドライブから取得（アップロード不要）:', size_mb(DEST))
else:
    from google.colab import files
    up = files.upload()   # KAKENの研究者JSONを選択
    assert up, 'ファイルが選択されていません'
    shutil.move(list(up)[0], DEST)
    print('アップロード完了:', size_mb(DEST))
    if os.path.isdir('/content/drive/MyDrive'):   # ドライブがあれば次回用に保存
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy(DEST, DRIVE)
        print('次回のためドライブにも保存:', DRIVE)

## 5. 集計＆PDF生成

4種類のPDFを作ります：
- `kaken_report_target.pdf` … A4縦1枚レポート（氏名なし）
- `kaken_stepup_hist_target.pdf` … 所要年数ヒストグラム（氏名なし）
- `kaken_stepup_gantt_target.pdf` … 大型**あり**群を大型初採択年でアラインしたガントチャート（**氏名あり**）
- `kaken_nolarge_gantt_target.pdf` … 大型**なし**群を翌年度でアラインしたガントチャート（**氏名あり**、「経過N年」付き）

In [ ]:
import sys
sys.argv = ['', KEY]

import kaken_stepup, kaken_report, kaken_stepup_gantt, kaken_nolarge_gantt

kaken_stepup.main()
kaken_report.main()
kaken_stepup_gantt.main()
kaken_nolarge_gantt.main()

### レポートを画面で確認

In [ ]:
import fitz  # PyMuPDF（poppler不要でPDFを画像化）
from IPython.display import Image, display
fitz.open(f'output/kaken_report_{KEY}.pdf')[0].get_pixmap(dpi=120).save('report_preview.png')
display(Image('report_preview.png'))

## 6. PDFをダウンロード

In [ ]:
from google.colab import files
for f in ['kaken_report', 'kaken_stepup_hist', 'kaken_stepup_gantt', 'kaken_nolarge_gantt']:
    files.download(f'output/{f}_{KEY}.pdf')